# Homework Starter — Stage 10b: Time Series & Classification
Fill in the TODOs. Use your own project dataset or adapt the synthetic generator below.

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas
# !pip install seaborn
# !pip install matplotlib
# !pip install scikit-learn

In [2]:
# Imports
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, TimeSeriesSplit
np.random.seed(7); sns.set(); plt.rcParams['figure.figsize']=(9,4)

## Option A: Use Your Project Data (Recommended)
Load your data here (ensure a DateTime index for time series).

In [3]:
from pathlib import Path
import pandas as pd
ROOT=Path.cwd()
for p in [ROOT,*ROOT.parents]:
 if (p/'project'/'data').exists(): ROOT=p; break
PROJECT_ROOT=ROOT/'project'
df=pd.read_csv(PROJECT_ROOT/'data/processed/sp500_sector_features_stage09.csv',parse_dates=['date'])
df=df.sort_values(['date','ticker']).reset_index(drop=True)
df['future_return_5d']=df.groupby('ticker')['close'].shift(-5)/df['close']-1
df.head()

,date,ticker,close,momentum_5d,daily_return,rolling_mean_return_20d,sector_Communication Services,sector_Consumer Discretionary,sector_Consumer Staples,sector_Energy,sector_Financials,sector_Health Care,sector_Industrials,sector_Information Technology,sector_Materials,sector_Real Estate,sector_Utilities,future_return_5d
0,2021-08-30 13:30:00,XLB,42.930000,NaN,NaN,NaN,0,0,0,0,0,0,0,0,1,0,0,-0.012229
1,2021-08-30 13:30:00,XLC,85.300003,NaN,NaN,NaN,1,0,0,0,0,0,0,0,0,0,0,0.004338
2,2021-08-30 13:30:00,XLE,24.365000,NaN,NaN,NaN,0,0,0,1,0,0,0,0,0,0,0,-0.008208
3,2021-08-30 13:30:00,XLF,38.450001,NaN,NaN,NaN,0,0,0,0,1,0,0,0,0,0,0,-0.015865
4,2021-08-30 13:30:00,XLI,104.800003,NaN,NaN,NaN,0,0,0,0,0,0,1,0,0,0,0,-0.019275


## Option B: Synthetic Generator (Use if you don't have data ready)

In [4]:
# Project data is loaded in the previous cell.

## Feature Engineering

In [5]:
df['lag_1']=df.groupby('ticker')['daily_return'].shift(1)
df['roll_mean_5']=df.groupby('ticker')['daily_return'].transform(lambda s:s.rolling(5).mean().shift(1))
df['roll_vol_20']=df.groupby('ticker')['daily_return'].transform(lambda s:s.rolling(20).std().shift(1))
df['y_next_ret']=df.groupby('ticker')['daily_return'].shift(-1)
df_feat=df.dropna().sort_values('date').copy()
df_feat.head()

,date,ticker,close,momentum_5d,daily_return,rolling_mean_return_20d,sector_Communication Services,sector_Consumer Discretionary,sector_Consumer Staples,sector_Energy,...,sector_Industrials,sector_Information Technology,sector_Materials,sector_Real Estate,sector_Utilities,future_return_5d,lag_1,roll_mean_5,roll_vol_20,y_next_ret
231,2021-09-29 13:30:00,XLB,40.180000,0.003873,-0.004090,-0.003111,0,0,0,0,...,0,0,1,0,0,0.002738,-0.012362,0.003704,0.010013,-0.015555
232,2021-09-29 13:30:00,XLC,80.360001,-0.007288,-0.000870,-0.003128,1,0,0,0,...,0,0,0,0,0,0.008337,-0.024381,-0.001310,0.008890,-0.003111
233,2021-09-29 13:30:00,XLE,26.445000,0.083589,0.000000,0.004659,0,0,0,1,...,0,0,0,0,0,0.029684,0.003415,0.022473,0.021366,-0.015126
234,2021-09-29 13:30:00,XLF,38.130001,0.027486,0.000787,-0.000284,0,0,0,0,...,0,0,0,0,0,0.012326,-0.016520,0.008716,0.012006,-0.015736
235,2021-09-29 13:30:00,XLI,99.889999,0.007768,0.001102,-0.002220,0,0,0,0,...,1,0,0,0,0,0.002903,-0.011590,0.002951,0.008935,-0.020523


## Split

In [6]:
cut=int(len(df_feat)*.8)
train,test=df_feat.iloc[:cut],df_feat.iloc[cut:]
features=['lag_1','roll_mean_5','roll_vol_20']
X_tr,X_te=train[features],test[features]
y_tr_reg,y_te_reg=train['y_next_ret'],test['y_next_ret']

## Pipeline + Model (Choose one track below)

In [7]:
# Track 1: Forecasting returns
reg = Pipeline([('scaler', StandardScaler()), ('linreg', LinearRegression())])
reg.fit(X_tr, y_tr_reg)
pred = reg.predict(X_te)
rmse = mean_squared_error(y_te_reg, pred) ** 0.5   # `squared=False` was removed in sklearn >= 1.6
print('RMSE:', rmse)

RMSE: 0.011227966626177364


In [8]:
# Classification is optional; this submission uses the time-series regression track.

## Interpretation

The time-aware split prevents future data from entering training. Lag and rolling features use historical values only. Financial returns are noisy, so RMSE is a baseline rather than a trading guarantee.